# Loan Default Prediction 

### ingestion

loading the raw data and verifying.

In [134]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("yasserh/loan-default-dataset")
file_path = os.path.join(path, "Loan_Default.csv")
data = pd.read_csv(file_path)

Validating the data

In [135]:
data.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [136]:
data.shape

(148670, 34)

In [137]:
data.columns

Index(['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type',
       'loan_purpose', 'Credit_Worthiness', 'open_credit',
       'business_or_commercial', 'loan_amount', 'rate_of_interest',
       'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization',
       'interest_only', 'lump_sum_payment', 'property_value',
       'construction_type', 'occupancy_type', 'Secured_by', 'total_units',
       'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type',
       'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type',
       'Status', 'dtir1'],
      dtype='str')

In [138]:
X = data.drop(columns=['Status','Gender','ID','year'])   
Y = data['Status']

### pre-processing (for feature selection tests)
Handles nulls and encoding

After doing the explanatory data analysis strategy for imputation of NaN values for numerical features which exhibits the skewness by median and for those numerical features, which does not exhibits skewness by median(since both mean and median will be the same, but to ensure consistency throughout the data media will be used for imputation)

For categorical features, the strategy for imputation (after doing the data analysis of the features with other features through cross tabulation and not finding a supporting relationship between them) by mode.


In [139]:
num_cols = [
    "rate_of_interest",
    "Interest_rate_spread",
    "Upfront_charges",
    "term",
    "property_value",
    "income",
    "LTV",
    "dtir1"
]

cat_cols = [
    "loan_limit",
    "approv_in_adv",
    "submission_of_application",
    "age",
    "loan_purpose",
    "Neg_ammortization"
]

# Numerical
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

# Categorical
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])



One-Hot Encoding categorical features.

In [140]:
Cat_cols = [col for col in X.columns if X[col].dtype in ['str', 'category']]
#One-Hot Encoding 

X = pd.get_dummies(data = X,
                         prefix = Cat_cols,
                         columns = Cat_cols)

### Feature Selection



Pearson Correlation Coefficients for Numerical Features

In [141]:
from scipy import stats
Num_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]

num_df = X[Num_cols]
pearson_dic = {i: stats.pearsonr(num_df[i], data['Status']) for i in num_df.columns}
pd.DataFrame(pearson_dic, index = ['Pearson Statistic', 'p-value']).T.sort_values(by='Pearson Statistic', ascending = False)

,Pearson Statistic,p-value
dtir1,0.082432,1.929235e-222
LTV,0.042656,7.788582e-61
Credit_Score,0.004004,1.226544e-01
term,-0.000207,9.363109e-01
loan_amount,-0.036825,8.690628e-46
rate_of_interest,-0.046738,1.119591e-72
Interest_rate_spread,-0.049536,2.031071e-81
income,-0.060618,4.882970e-121
property_value,-0.080905,2.522136e-214
Upfront_charges,-0.095094,1.210288e-295


Chi-Square test of Categorical Variables¶


In [142]:

def chi_square_test(cat_var, target_var):

    contingency_table = pd.crosstab(data[cat_var], data[target_var])
    chi2, p, _, _ = stats.chi2_contingency(contingency_table)
    return chi2, p, contingency_table


chi_square_results = []


for col in Cat_cols:
    chi2, p, contingency_table = chi_square_test(col, 'Status')
    chi_square_results.append({
        'Variable': col,
        'Chi2 Statistic': chi2,
        'P-Value': p,
        'Contingency Table': contingency_table
    })


chi_square_results_df = pd.DataFrame(chi_square_results)


chi_square_results_df.sort_values(by='Chi2 Statistic', ascending = False)

,Variable,Chi2 Statistic,P-Value,Contingency Table
14,credit_type,52135.280705,0.000000e+00,Status 0 1 credit_type ...
9,lump_sum_payment,5237.827442,0.000000e+00,Status 0 1 lump_sum_payme...
7,Neg_ammortization,3610.208104,0.000000e+00,Status 0 1 Neg_ammortiza...
15,co-applicant_credit_type,3092.392571,0.000000e+00,Status 0 1 co-appl...
17,submission_of_application,2171.709755,0.000000e+00,Status 0 1 submis...
2,loan_type,1309.958143,3.517253e-285,Status 0 1 loan_type ...
6,business_or_commercial,1272.807998,9.172191e-279,Status 0 1 business_...
0,loan_limit,427.398487,5.985647e-95,Status 0 1 loan_limit ...
18,Region,380.456330,3.786057e-82,Status 0 1 Region ...
16,age,374.501644,8.442234e-78,Status 0 1 age 25-34 ...


Mutual Information (for Categorical and Numerical Features)


In [143]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X, Y, discrete_features = 'auto')
mi_scores_df = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print("Mutual Information Scores:")
print(mi_scores_df.to_string())

Mutual Information Scores:
Interest_rate_spread                  0.559007
Upfront_charges                       0.478047
rate_of_interest                      0.348858
LTV                                   0.174665
credit_type_EQUI                      0.163451
property_value                        0.140701
dtir1                                 0.085117
lump_sum_payment_not_lpsm             0.032413
open_credit_nopc                      0.032020
construction_type_sb                  0.031152
Neg_ammortization_not_neg             0.029847
Secured_by_home                       0.029234
Security_Type_direct                  0.028971
total_units_1U                        0.028000
submission_of_application_to_inst     0.027036
co-applicant_credit_type_CIB          0.026385
interest_only_not_int                 0.025981
co-applicant_credit_type_EXP          0.023804
loan_type_type1                       0.023714
income                                0.023548
Credit_Worthiness_l1             

Final selected feature for model training

In [144]:
num_features = ['income', 'loan_amount', 'dtir1', 'LTV', 'property_value']


cat_features = ['loan_limit', 'approv_in_adv', 'loan_type', 'loan_purpose', 
               'Credit_Worthiness', 'business_or_commercial', 'Neg_ammortization',
               'interest_only', 'lump_sum_payment', 'occupancy_type', 'credit_type',
               'co-applicant_credit_type', 'Region']

features = num_features + cat_features

In [145]:
from sklearn.model_selection import train_test_split

X_t = data[features].copy()
Y_t = data['Status']

# train-test split
X_train, x_test, y_train, Y_test = train_test_split(X_t, Y_t, train_size = 0.6, random_state = 0)
X_test,x_validate,y_test,y_validate = train_test_split(x_test, Y_test, train_size = 0.5, random_state = 0)


### Model Building

Building a random forest model

In [146]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


# Preprocessing transformers

categorical_columns = [col for col in X_train.columns if X_train[col].dtype in ['category', 'object']]
numerical_columns = [col for col in X_train.columns if X_train[col].dtype in ['int64', 'float64']]

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])


preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_columns),
    ('cat', cat_transformer, categorical_columns)
])

In [147]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, min_samples_split = 2, max_depth = 10, random_state=0)

rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf)
])

rf_pipe.fit(X_train, y_train)

# Prediction
y_pred = rf_pipe.predict(X_test)

# Evaluation
print(f"Model Accuracy: {rf_pipe.score(X_test, y_test)}\n")

print(f"Confusion Matrix: \n {confusion_matrix(y_test, y_pred)}\n")
print(f"Classification Report : \n {classification_report(y_test, y_pred)}\n")


print(f"ROC-AUC Score: \n {roc_auc_score(y_test, rf_pipe.predict_proba(X_test)[:, 1])}")


Model Accuracy: 0.8561915652115424

Confusion Matrix: 
 [[20909  1519]
 [ 2757  4549]]

Classification Report : 
               precision    recall  f1-score   support

           0       0.88      0.93      0.91     22428
           1       0.75      0.62      0.68      7306

    accuracy                           0.86     29734
   macro avg       0.82      0.78      0.79     29734
weighted avg       0.85      0.86      0.85     29734


ROC-AUC Score: 
 0.8441022098955243


In [ ]:
from sklearn.model_selection import RandomizedSearchCV


params = {
    'classifier__n_estimators': [100, 200, 300],   
    'classifier__max_depth': [5, 10, None],          
    'classifier__min_samples_split': [2, 5, 10],          
    'classifier__min_samples_leaf': [1, 2, 4],        
    'classifier__max_features': ['sqrt', 'log2']  
}


random_search = RandomizedSearchCV(rf_pipe, param_distributions=params, 
                                   n_iter=50, cv=5, scoring='roc_auc', verbose=2, n_jobs=-1, random_state=0)


random_search.fit(X_train, y_train)

# Get the best parameters and model
best_model_rf = random_search.best_estimator_
print(random_search.best_params_)

# Evaluating the best model
y_pred_best = best_model_rf.predict(X_test)
print(f"Best Model Accuracy: {best_model_rf.score(X_test, y_test)}\n")

print(f"Classification Report : \n{classification_report(y_test, y_pred_best)}\n")

roc_auc = roc_auc_score(y_test, best_model_rf.predict_proba(X_test)[:, 1])
print(f"ROC-AUC Score: {roc_auc}")


XGBoost model

In [148]:
categorical_columns = [col for col in X_train.columns if X_train[col].dtype in ['category', 'object']]
numerical_columns = [col for col in X_train.columns if X_train[col].dtype in ['int64', 'float64']]


# Preprocessing transformers

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

# Num_transformer with scaler
"""
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
"""
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])



preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_columns),
    ('cat', cat_transformer, categorical_columns)
])

In [149]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

xgb_pipeline.fit(X_train, y_train)


# Prediction
y_pred = xgb_pipeline.predict(X_test)

# Evaluation
print(f"Model Accuracy: {xgb_pipeline.score(X_test, y_test)}\n")

print(f"Confusion Matrix: \n {confusion_matrix(y_test, y_pred)}\n")
print(f"Classification Report : \n {classification_report(y_test, y_pred)}\n")


print(f"ROC-AUC Score: \n {roc_auc_score(y_test, xgb_pipeline.predict_proba(X_test)[:, 1])}")

Model Accuracy: 0.8725701217461492

Confusion Matrix: 
 [[22024   404]
 [ 3385  3921]]

Classification Report : 
               precision    recall  f1-score   support

           0       0.87      0.98      0.92     22428
           1       0.91      0.54      0.67      7306

    accuracy                           0.87     29734
   macro avg       0.89      0.76      0.80     29734
weighted avg       0.88      0.87      0.86     29734


ROC-AUC Score: 
 0.839816396256078


/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [150]:
from sklearn.model_selection import RandomizedSearchCV

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.01, 0.1],
    'classifier__max_depth': [3, 7, 10]
}
random_search = RandomizedSearchCV(xgb_pipeline, param_grid, cv=5, scoring='roc_auc',
                                  n_jobs=-1, random_state=0)

random_search.fit(X_train, y_train)


# Get the best parameters and model
best_model_xgb = random_search.best_estimator_
print(random_search.best_params_)

# Evaluating the best model
y_pred_best = best_model_xgb.predict(X_test)
print(f"Best Model Accuracy: {best_model_xgb.score(X_test, y_test)}\n")

print(f"Classification Report : \n{classification_report(y_test, y_pred_best)}\n")

roc_auc = roc_auc_score(y_test, best_model_xgb.predict_proba(X_test)[:, 1])
print(f"ROC-AUC Score: {roc_auc}")


/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:53] WARNING:

{'classifier__n_estimators': 200, 'classifier__max_depth': 10, 'classifier__learning_rate': 0.01}
Best Model Accuracy: 0.8722001748839712

Classification Report : 
              precision    recall  f1-score   support

           0       0.86      0.99      0.92     22428
           1       0.94      0.51      0.66      7306

    accuracy                           0.87     29734
   macro avg       0.90      0.75      0.79     29734
weighted avg       0.88      0.87      0.86     29734


ROC-AUC Score: 0.8436489786753693


Model cross validation

In [151]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model_xgb, X_t, Y_t, cv=5, scoring='roc_auc')
print(f"Cross-validated AUC scores: {cv_scores}")
print(f"Mean AUC Score across folds: {cv_scores.mean()}")

/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vedant/Coding/Loan-Default-Prediction-/env/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [00:48:59] WARNING:

Cross-validated AUC scores: [0.84321562 0.84257319 0.83650231 0.83536897 0.83745191]
Mean AUC Score across folds: 0.8390223991066492


### Feature importance

xgb_model = random_search.best_estimator_.named_steps['classifier']

# Feature names from the preprocessor
cat_features = random_search.best_estimator_.named_steps['preprocessor'].transformers_[1][1].named_steps['one_hot'].get_feature_names_out()
num_features = random_search.best_estimator_.named_steps['preprocessor'].transformers_[0][1].named_steps['imputer'].get_feature_names_out()
all_features = list(num_features) + list(cat_features)

feature_importances = pd.Series(xgb_model.feature_importances_, index=all_features).sort_values(ascending=False)

print(feature_importances)


### Model validation

In [152]:
from sklearn.metrics import accuracy_score

# Prediction

y_pred = best_model_xgb.predict(x_validate)

# Evaluation
print(f"Model Accuracy: {accuracy_score(y_validate, y_pred)}\n")

print(f"Confusion Matrix: \n {confusion_matrix(y_validate, y_pred)}\n")
print(f"Classification Report : \n {classification_report(y_validate, y_pred)}\n")
print(f"ROC-AUC Score: \n {roc_auc_score(y_validate, best_model_xgb.predict_proba(x_validate)[:, 1])}")




Model Accuracy: 0.8716620703571669

Confusion Matrix: 
 [[22249   247]
 [ 3569  3669]]

Classification Report : 
               precision    recall  f1-score   support

           0       0.86      0.99      0.92     22496
           1       0.94      0.51      0.66      7238

    accuracy                           0.87     29734
   macro avg       0.90      0.75      0.79     29734
weighted avg       0.88      0.87      0.86     29734


ROC-AUC Score: 
 0.8363464609790199


### Model deployment

In [153]:
import joblib

# Save the model
joblib.dump(best_model_xgb, 'xgboost_model.pkl')

['xgboost_model.pkl']

In [154]:
import numpy as np
# batch predictions

new_data = {'ID': {0: 170054, 1: 1244, 2: 33228, 3: 75396, 4: 13660, 5: 54176, 6: 90586, 7: 13853, 8: 9158, 9: 13863},
 'year': {0: 2019, 1: 2019, 2: 2019, 3: 2019, 4: 2019, 5: 2019, 6: 2019, 7: 2019, 8: 2019, 9: 2019},
 'loan_limit': {0: 'cf', 1: 'cf', 2: 'cf', 3: 'cf', 4: 'cf', 5: 'cf', 6: 'cf', 7: 'cf', 8: 'cf', 9: 'cf'},
 'Gender': {0: 'Joint', 1: 'Male', 2: 'Male', 3: 'Sex Not Available', 4: 'Male', 5: 'Sex Not Available', 6: 'Female',
  7: 'Male', 8: 'Sex Not Available', 9: 'Sex Not Available'},
 'approv_in_adv': {0: 'pre', 1: 'nopre', 2: 'nopre', 3: 'pre', 4: 'nopre', 5: 'nopre', 6: 'nopre', 7: 'nopre', 8: 'nopre', 9: 'nopre'},
 'loan_type': {0: 'type2', 1: 'type1', 2: 'type1', 3: 'type1', 4: 'type2', 5: 'type1', 6: 'type3', 7: 'type1', 8: 'type1', 9: 'type1'},
 'loan_purpose': {0: 'p1', 1: 'p3', 2: 'p4', 3: 'p4', 4: 'p3', 5: 'p3', 6: 'p1', 7: 'p1', 8: 'p2', 9: 'p4'},
 'Credit_Worthiness': {0: 'l1', 1: 'l1', 2: 'l1', 3: 'l1', 4: 'l1', 5: 'l1', 6: 'l1', 7: 'l1', 8: 'l1', 9: 'l1'},
 'open_credit': {0: 'nopc', 1: 'nopc', 2: 'nopc', 3: 'nopc', 4: 'nopc', 5: 'nopc', 6: 'nopc', 7: 'nopc', 8: 'nopc', 9: 'nopc'},
 'business_or_commercial': {0: 'b/c', 1: 'nob/c', 2: 'nob/c', 3: 'nob/c', 4: 'b/c', 5: 'nob/c', 6: 'nob/c', 
                            7: 'nob/c', 8: 'nob/c',  9: 'nob/c'},
 'loan_amount': {0: 486500,  1: 456500,  2: 706500,  3: 196500,  4: 346500,  5: 146500,  6: 616500,  7: 196500,  8: 106500,  9: 496500},
 'rate_of_interest': {0: 4.125,  1: np.nan,  2: 3.25,  3: 3.5,  4: 3.49,  5: 4.875,  6: 3.125,  7: np.nan,  8: np.nan,  9: 3.99},
 'Interest_rate_spread': {0: 0.6687,  1: np.nan,  2: -0.3444,  3: -0.1606,  4: 0.3966,  5: 1.3592,  6: -0.663,  7: np.nan,  8: np.nan,  9: 0.4232},
 'Upfront_charges': {0: 1052.73,  1: np.nan,  2: 7024.0,  3: 343.75,  4: 0.0,  5: 1250.0,  6: 6100.0,  7: np.nan,  8: np.nan,  9: 0.0},
 'term': {0: 360.0,  1: 360.0,  2: 360.0,  3: 360.0,  4: 360.0,  5: 360.0,  6: 360.0,  7: 360.0,  8: 180.0,  9: 360.0},
 'Neg_ammortization': {0: 'not_neg',  1: 'not_neg', 2: 'not_neg', 3: 'not_neg', 4: 'not_neg', 5: 'not_neg',
                       6: 'not_neg', 7: 'not_neg', 8: 'not_neg', 9: 'neg_amm'},
 'interest_only': {0: 'not_int', 1: 'not_int', 2: 'not_int', 3: 'not_int', 4: 'not_int', 5: 'not_int',
                   6: 'not_int', 7: 'not_int', 8: 'not_int', 9: 'not_int'},
 'lump_sum_payment': {0: 'not_lpsm', 1: 'not_lpsm', 2: 'not_lpsm', 3: 'not_lpsm', 4: 'not_lpsm', 5: 'not_lpsm',
                      6: 'not_lpsm', 7: 'not_lpsm', 8: 'lpsm', 9: 'not_lpsm'},
 'property_value': {0: 498000.0, 1: 668000.0, 2: 948000.0, 3: 1458000.0, 4: 428000.0, 5: 248000.0, 6: 618000.0,
                     7: np.nan, 8: 268000.0, 9: 578000.0},
 'construction_type': {0: 'sb', 1: 'sb', 2: 'sb', 3: 'sb', 4: 'sb', 5: 'sb', 6: 'sb', 7: 'sb', 8: 'sb', 9: 'sb'},
 'occupancy_type': {0: 'pr', 1: 'ir', 2: 'pr', 3: 'pr', 4: 'pr', 5: 'sr', 6: 'pr', 7: 'pr', 8: 'pr', 9: 'pr'},
 'Secured_by': {0: 'home', 1: 'home', 2: 'home', 3: 'home', 4: 'home', 5: 'home', 6: 'home', 7: 'home', 8: 'home', 9: 'home'},
 'total_units': {0: '1U', 1: '1U', 2: '1U', 3: '1U', 4: '1U', 5: '1U', 6: '1U', 7: '1U', 8: '1U', 9: '1U'},
 'income': {0: 7860.0, 1: 780.0,  2: 12780.0, 3: 11820.0, 4: 3780.0, 5: 4560.0,
            6: 10920.0, 7: 3780.0, 8: 0.0, 9: 6180.0},
 'credit_type': {0: 'CRIF', 1: 'CRIF', 2: 'CIB', 3: 'CRIF', 4: 'EXP', 5: 'EXP', 6: 'EXP', 7: 'EQUI', 8: 'CRIF', 9: 'EXP'},
 'Credit_Score': {0: 741, 1: 849, 2: 731, 3: 742, 4: 522, 5: 858, 6: 680, 7: 602, 8: 886, 9: 674},
 'co-applicant_credit_type': {0: 'EXP', 1: 'CIB', 2: 'CIB', 3: 'CIB', 4: 'CIB', 5: 'EXP', 6: 'CIB', 7: 'EXP', 8: 'CIB', 9: 'CIB'},
 'age': {0: '35-44', 1: '35-44', 2: '35-44', 3: '45-54', 4: '65-74',
         5: '55-64', 6: '35-44', 7: '35-44', 8: '45-54', 9: '35-44'},
 'submission_of_application': {0: 'not_inst', 1: 'not_inst', 2: 'not_inst', 3: 'to_inst',
                               4: 'not_inst', 5: 'to_inst', 6: 'not_inst', 7: 'to_inst', 8: 'to_inst', 9: 'not_inst'},
 'LTV': {0: 97.69076305, 1: 68.33832335, 2: 74.52531646, 3: 13.47736626, 4: 80.95794393, 
         5: 59.07258065, 6: 99.75728155, 7: np.nan, 8: 39.73880597, 9: 85.89965398},
 'Region': {0: 'North', 1: 'North', 2: 'North', 3: 'south', 4: 'central', 5: 'south',
             6: 'North', 7: 'North', 8: 'south', 9: 'south'},
 'Security_Type': {0: 'direct', 1: 'direct', 2: 'direct', 3: 'direct', 4: 'direct',
                   5: 'direct', 6: 'direct', 7: 'direct', 8: 'direct', 9: 'direct'},
 'dtir1': {0: 42.0, 1: 13.0, 2: 32.0, 3: 42.0, 4: 55.0, 5: 36.0, 6: 29.0, 7: np.nan, 8: 18.0, 9: 46.0}}

new_data_df = pd.DataFrame(new_data)


In [155]:
loaded_model = joblib.load('xgboost_model.pkl')

predictions = loaded_model.predict(new_data_df)

print(predictions)

[0 1 0 0 0 0 0 1 1 0]
